# Teaching Agent — RAG Retrieval Prototype (ChromaDB)

End-to-end demo of the RAG pipeline:

1. **Extract** mixed-format docs (pdf/docx/pptx/md/img + OCR) → raw text
2. **Chunk** structure-aware (section-coherent, page-tracked)
3. **JSON artifacts** written before *and* after chunking (inspectable)
4. **Embed + store** in a local persistent ChromaDB collection
5. **Retrieve** by semantic similarity, with topic filtering + scores

No server, no Docker — Chroma persists to `./chroma_db`, embeddings run locally.

In [ ]:
# Make src importable when running from the repo root
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 1. Ingest — produces JSON artifacts + loads ChromaDB

Point at your teaching corpus. `--reset` clears the collection first.
This writes `artifacts/teaching_extracted.json` (before) and `artifacts/teaching_chunks.json` (after).

In [ ]:
from copilot.rag.ingest import ingest

ingest('teaching', '../data/teaching', reset=True)

## 2. Inspect the JSON artifacts

### Before chunking — raw extraction with provenance

In [ ]:
import json

extracted = json.load(open('../artifacts/teaching_extracted.json', encoding='utf-8'))
print(f'{len(extracted)} documents extracted')
doc = extracted[0]
print('source :', doc['source'])
print('topic  :', doc['topic'])
print('blocks :', doc['num_blocks'])
print('first block:', doc['blocks'][0])

### After chunking — chunks with metadata (what gets embedded)

In [ ]:
chunks = json.load(open('../artifacts/teaching_chunks.json', encoding='utf-8'))
print(f'{len(chunks)} chunks total')
chunks[0]

## 3. Retrieval prototype

Semantic search over the Chroma collection. Each hit shows source, topic, page, section, and a similarity score.

In [ ]:
from copilot.rag.retriever import retrieve, format_passages, list_topics

print('Topics in the collection:')
for topic, n in list_topics('teaching'):
    print(f'  {topic}: {n} chunks')

In [ ]:
query = 'How does the CORE principle structure modules?'
passages = retrieve('teaching', query, k=5)
print(format_passages(passages))

### Topic-filtered retrieval
Scope the search to a single topic folder — the key lever for precision on a broad corpus.

In [ ]:
passages = retrieve(
    'teaching',
    'active learning and reflection methods',
    k=5,
    topics=['03_Didactics_and_Instructional_Design'],
)
print(format_passages(passages))

### Raw results as data (for downstream use / the future agentic layer)

In [ ]:
from dataclasses import asdict

results = retrieve('teaching', 'responsible use of generative AI', k=3)
[asdict(p) for p in results]

## 4. Full RAG — retrieve + generate (local Ollama LLM)

This is the complete RAG loop: retrieve grounded chunks, then have the local
open-source LLM write a cited answer. Requires Ollama running (`ollama serve`).
If Ollama isn't up, retrieval still works and you'll get the passages back.

In [ ]:
from copilot.rag.generate import answer, _format

result = answer('teaching', 'How does the CORE principle structure modules?')
print(_format(result))